In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd


# Load the dataset
steamGamesFile = "./data/steam_data.csv"
steamGames = pd.read_csv(steamGamesFile)


# Display the title
st.title("Stinder - Steam Game Recommender")


# Display the dataset
st.write("Here is the dataset of Steam games:")
st.write(steamGames.head())


# User input for game recommendation
user_input = st.text_input("Enter a game you like:")


# Simple recommendation logic (for demonstration purposes)
if user_input:
    recommendations = steamGames[steamGames['name'].str.contains(user_input, case=False)]
    st.write("Recommended games:")
    st.write(recommendations)

In [ ]:
from nltk.tokenize import word_tokenize 
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') 
##nltk.download()

ps = PorterStemmer()

preprocessedText = []

for row in steamGames.itertuples():
    
    
    text = word_tokenize(str(row[4])) ## indice de la columna que contiene el texto
    ## Remove stop words
    stops = set(stopwords.words("english"))
    text = [ps.stem(w) for w in text if not w in stops and w.isalnum()]
    text = " ".join(text)
    
    preprocessedText.append(text)

processedSteamGames = steamGames
processedSteamGames['processed_text'] = preprocessedText
processedSteamGames


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

bagOfWordsModel = TfidfVectorizer()
bagOfWordsModel.fit(processedSteamGames['processed_text'])
textsBoW= bagOfWordsModel.transform(processedSteamGames['processed_text'])
print("Finished")

In [ ]:
textsBoW.shape

In [ ]:
print(textsBoW)

In [ ]:
bagOfWordsModel.get_feature_names_out()

In [7]:
from sklearn.metrics import pairwise_distances

distance_matrix= pairwise_distances(textsBoW, metric='cosine')

In [ ]:
print(distance_matrix.shape)
print(type(distance_matrix))

In [ ]:
from fuzzywuzzy import process

# Función para buscar el appid por nombre de juego y mostrar los resultados
def search_game_by_name(game_name):
    # Obtener solo la columna de nombres
    game_names = processedSteamGames['name'].tolist()

    # Buscar las coincidencias más cercanas usando fuzzywuzzy
    matches = process.extract(game_name, game_names, limit=10)  # Limitar a las 10 mejores coincidencias

    # Filtrar las coincidencias que tienen un puntaje alto
    best_matches = [(match[0], match[1]) for match in matches if match[1] > 70]  # Umbral de 70 para coincidencias

    if best_matches:
        print("Se encontraron las siguientes coincidencias:")
        for i, (name, score) in enumerate(best_matches):
            print(f"{i}: {name} (Puntaje: {score})")
        return best_matches  # Retorna las coincidencias encontradas
    else:
        print("No se encontraron coincidencias.")
        return []
    
# Ejemplo de uso
game_name_to_search = input("Ingresa el nombre del videojuego a buscar: ")
best_matches = search_game_by_name(game_name_to_search)


In [ ]:
def select_game(best_matches):
    """
    Permite al usuario seleccionar un juego de entre las mejores coincidencias y retorna el índice seleccionado.
    Si hay solo un juego, lo selecciona automáticamente.
    """
    selected_index = None
    if best_matches:
        if len(best_matches) > 1:
            # Permitir al usuario seleccionar una opción si hay varias coincidencias
            try:
                selected_index = int(input(f"Selecciona el número del juego que deseas (0-{len(best_matches)-1}): "))
                selected_game_name = best_matches[selected_index][0]
                selected_game = processedSteamGames[processedSteamGames['name'] == selected_game_name]
                print(f"Has seleccionado: {selected_game_name} (appid: {selected_game['appid'].values[0]})")
            except (ValueError, IndexError):
                print("Selección inválida.")
                return None  # Salir sin selección válida
        else:
            # Si hay solo un juego, seleccionarlo automáticamente
            selected_game_name = best_matches[0][0]
            selected_game = processedSteamGames[processedSteamGames['name'] == selected_game_name]
            print(f"El único juego encontrado es: {selected_game_name} (appid: {selected_game['appid'].values[0]})")
            selected_index = 0
    else:
        print("No hay juegos para seleccionar.")
    return selected_index


# Si hay resultados en las mejores coincidencias
if best_matches:
    selected_index = select_game(best_matches)  # Obtener índice del juego seleccionado
    
    if selected_index is not None:
        # Obtener el nombre del juego seleccionado
        selected_game_name = best_matches[selected_index][0]
        
        # Calcular el índice del título en el DataFrame original
        indexOfTitle = processedSteamGames[processedSteamGames['name'] == selected_game_name].index[0]
        
        # Calcular la distancia y seleccionar las mejores coincidencias
        distance_scores = list(enumerate(distance_matrix[indexOfTitle]))
        ordered_scores = sorted(distance_scores, key=lambda x: x[1])
        top_scores = ordered_scores[1:11]  # Excluir el propio juego (índice 0)
        top_indexes = [i[0] for i in top_scores]
        
        # Mostrar los nombres de los juegos similares
        similar_games = processedSteamGames['name'].iloc[top_indexes]
        print("Juegos similares encontrados:")
        print(similar_games)
else:
    print("No se encontraron coincidencias.")
